In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch.nn.functional as F
import time
ENTRY_BYTE_SIZE = 487
ADDITIONAL_COLORED_FEATS = 24
NUM_COLORLESS = ENTRY_BYTE_SIZE - 2
READ_SIZE = ENTRY_BYTE_SIZE + ADDITIONAL_COLORED_FEATS
torch.set_float32_matmul_precision('high')

In [ ]:
# Add more files if needed.
with open('out.bin', 'rb') as f: # Lichess big 3
    feats = f.read()
    feats = np.frombuffer(feats, dtype=np.int8)
# Reshape the features to have the correct shape. Do it not as a view, but as a copy.
feats = feats.reshape(-1, READ_SIZE).copy()

with open('out1.bin', 'rb') as f: # Ethereal FRC cleaned up
    feats2 = f.read()
    feats2 = np.frombuffer(feats2, dtype=np.int8)
# Reshape the features to have the correct shape. Do it not as a view, but as a copy.
feats2 = feats2.reshape(-1, READ_SIZE).copy()

# ...

features = np.concatenate([feats, feats2])
print(features.shape)

del feats
del feats2

In [ ]:
# Shuffle the features.
np.random.seed(47) # 47 my beloved.
np.random.shuffle(features)
torch.manual_seed(47) # 47 my sweethearth.

In [ ]:
# Divide the features into training and testing sets.
train_size = int(0.94 * len(features))


phases_train = torch.tensor(features[:train_size,0], dtype=torch.float16).unsqueeze(1)
phases_test  = torch.tensor(features[train_size:,0], dtype=torch.float16).unsqueeze(1)

colorless_train = torch.tensor(features[:train_size, 1:ENTRY_BYTE_SIZE-1], dtype=torch.int8)
colorless_test  = torch.tensor(features[train_size:, 1:ENTRY_BYTE_SIZE-1], dtype=torch.int8)

colored_train = torch.tensor(features[:train_size, ENTRY_BYTE_SIZE-1:-1], dtype=torch.int8)
colored_test  = torch.tensor(features[train_size:, ENTRY_BYTE_SIZE-1:-1], dtype=torch.int8)

y_train = torch.tensor(features[:train_size, -1], dtype=torch.float16).unsqueeze(1)
y_test  = torch.tensor(features[train_size:, -1], dtype=torch.float16).unsqueeze(1)

phases_train /= 24
phases_test /= 24
y_train = (y_train + 1)/2 # -1,0,1 -> 0,0.5,1
y_test = (y_test + 1)/2

print(phases_train.shape)
print(colorless_train.shape)
print(colored_train.shape)
print(y_train.shape)

print(phases_test.shape)
print(colorless_test.shape)
print(colored_test.shape)
print(y_test.shape)

del features

In [ ]:
default = [
  103, 389, 399, 564, 883, 
       0,    0,    0,    0,    0,    0,    0,    0 ,
      66,   24,   65,   83,   71,   52,  -49,  -43 ,
       5,   44,   62,   52,   60,   89,   62,    7 ,
       2,   -1,   14,   21,   30,   23,   -2,   -1 ,
      -7,  -12,    3,   19,   17,   15,   -8,   -5 ,
      -4,    1,    3,   10,   20,    5,   13,    1 ,
      -6,    0,   -1,   -2,    1,   12,   17,  -10 ,
       0,    0,    0,    0,    0,    0,    0,    0 ,

    -135,  -84,  -68,  -17,    7, -101, -102, -106 ,
      16,   17,   51,  104,   62,   94,   -2,   28 ,
      28,   71,   76,   74,  105,   82,   71,   20 ,
      43,   58,   60,   74,   66,   78,   53,   47 ,
      45,   51,   53,   58,   72,   50,   64,   52 ,
       9,   31,   42,   52,   56,   49,   44,   33 ,
      15,    3,   24,   39,   42,   31,   21,   34 ,
     -14,   10,   24,   22,   27,   27,   17,   -6 ,

     -23,  -47,  -76, -108,  -88, -121,  -23,  -39 ,
     -23,   26,   30,    0,   22,   16,    9,  -22 ,
      34,   54,   58,   46,   47,   56,   56,   37 ,
      28,   54,   49,   65,   69,   48,   52,   20 ,
      32,   32,   37,   60,   65,   43,   41,   30 ,
      33,   52,   51,   54,   54,   58,   58,   48 ,
      45,   48,   47,   35,   42,   47,   67,   53 ,
      52,   40,   35,   32,   27,   35,   34,   57 ,

      56,   44,   27,   21,   19,   37,   66,   77 ,
      48,   41,   60,   69,   51,   72,   52,   64 ,
      46,   97,   72,   90,  103,   86,  118,   67 ,
      46,   77,   81,   94,   85,   84,   91,   67 ,
      17,   34,   33,   39,   32,   27,   44,   28 ,
       8,   37,   27,   31,   33,   27,   58,   27 ,
     -16,   25,   34,   38,   38,   46,   50,   -7 ,
      27,   37,   42,   49,   46,   50,   49,   39 ,

      51,   64,   48,   41,   28,   52,   56,   46 ,
      57,   11,   40,  -18,  -40,   34,   -5,   59 ,
      62,   54,   41,   25,   19,   37,   54,   64 ,
      67,   75,   61,   49,   39,   53,   72,   69 ,
      77,   79,   60,   68,   69,   64,   81,   81 ,
      80,   92,   85,   79,   84,   88,   98,   89 ,
      78,   95,  102,  102,  100,  108,  100,   76 ,
      85,   97,  101,  109,  105,  101,  103,   88 ,

     -19,   33,   35,    9,   27,   28,   56,   -6 ,
     -53,   44,   55,  118,   82,  113,   54,   15 ,
     -33,  122,  124,   97,  110,  170,  111,   17 ,
     -43,   69,   98,   58,   82,  103,   73,  -62 ,
     -45,   69,   96,   60,   65,   92,   56,  -69 ,
     -42,   35,   64,   48,   57,   53,   33,  -48 ,
       4,    6,   15,  -29,  -18,  -10,   18,    2 ,
     -31,  -11,  -33,  -84,  -49,  -76,   -9,  -15 ,

-29, -15, -1, 8, 17, 24, 34, 45, 56,
-53, -12, 11, 19, 28, 33, 36, 37, 39, 44, 54, 78, 85, 105,
-9, -15, -4, -2, 0, 0, 3, 10, 14, 22, 22, 29, 30, 47, 93,
-83, -40, 2, 26, 30, 36, 44, 50, 55, 62, 67, 74, 78, 81, 82, 84, 86, 82, 84, 94, 100, 112, 110, 128, 118, 116, 104, 99,
11, 18, 2, 16, 1, 10, 4, 1,
0, -2, -29, -23, 14, 24, 94,
-2, 30, 31, 23, 32, 26, 13, 33, 34, 28, 41, 1, 63, 28, 24,

159, 451, 474, 782, 862,

       0,    0,    0,    0,    0,    0,    0,    0 ,
      89,   99,   65,   17,   18,   43,  102,  107 ,
      83,   79,   41,    7,    4,   28,   67,   77 ,
      43,   38,   19,    0,   -1,   12,   35,   34 ,
      17,   17,    4,   -6,   -8,    2,   11,   10 ,
      10,    8,   11,    7,    7,   12,    4,    1 ,
      18,   18,   28,   27,   25,   27,   12,    8 ,
       0,    0,    0,    0,    0,    0,    0,    0 ,

     -33,   34,   71,   55,   54,   70,   50,  -53 ,
      18,   47,   50,   57,   67,   44,   47,   15 ,
      40,   47,   87,   90,   80,   84,   45,   44 ,
      52,   73,   99,  107,  111,   93,   86,   56 ,
      59,   65,   87,   94,   96,   90,   68,   57 ,
      27,   39,   63,   78,   79,   63,   42,   22 ,
      39,   48,   37,   52,   52,   36,   39,   42 ,
      14,   26,   27,   39,   44,   33,   29,    4 ,

      76,   75,   73,   80,   82,   73,   66,   69 ,
      65,   70,   67,   64,   56,   69,   73,   65 ,
      73,   71,   71,   63,   65,   73,   70,   73 ,
      64,   69,   69,   82,   80,   73,   68,   68 ,
      49,   60,   80,   78,   79,   75,   59,   50 ,
      47,   61,   69,   71,   73,   67,   55,   48 ,
      43,   37,   50,   64,   65,   54,   42,   37 ,
      39,   55,   64,   50,   57,   67,   57,   32 ,

     141,  146,  154,  152,  153,  150,  143,  137 ,
     134,  144,  142,  138,  145,  136,  139,  129 ,
     128,  109,  122,  113,  107,  115,  101,  119 ,
     120,  113,  118,  109,  109,  110,  110,  112 ,
     104,  115,  114,  110,  111,  117,  113,  102 ,
      84,   96,   95,   93,   91,   95,   87,   71 ,
      82,   82,   86,   83,   80,   75,   66,   70 ,
      83,   83,   86,   78,   75,   85,   77,   69 ,

     128,  135,  163,  176,  186,  168,  147,  137 ,
     148,  198,  198,  239,  269,  203,  214,  158 ,
     160,  163,  196,  213,  207,  207,  170,  179 ,
     161,  182,  187,  213,  221,  196,  191,  166 ,
     131,  163,  177,  203,  192,  180,  151,  138 ,
     103,  121,  151,  147,  140,  152,  125,   94 ,
      91,   79,   80,  106,  110,   70,   73,   77 ,
      70,   67,   72,  101,   77,   52,   60,   50 ,

    -142,  -58,  -38,   -3,  -21,  -12,  -19, -147 ,
     -17,   43,   35,   11,   24,   37,   57,  -21 ,
      16,   51,   47,   45,   44,   44,   57,   14 ,
       6,   33,   45,   52,   47,   43,   37,    8 ,
     -23,    6,   25,   43,   41,   23,    8,  -16 ,
     -17,   -8,    7,   26,   22,    7,   -9,  -17 ,
     -21,   -6,    3,   15,   12,   10,  -11,  -29 ,
     -81,  -33,   -6,  -18,  -27,   -5,  -33,  -93 ,

-86, -21, 18, 40, 50, 64, 63, 59, 40,
-136, -72, -16, 12, 30, 46, 58, 63, 70, 70, 64, 51, 56, 22,
-53, -2, 50, 61, 81, 100, 108, 108, 114, 119, 124, 126, 130, 115, 96,
-107, -167, -105, -54, 41, 87, 118, 136, 153, 159, 163, 168, 169, 175, 178, 178, 179, 184, 181, 173, 167, 150, 148, 131, 131, 117, 119, 128,
53, 20, 18, 20, 4, 25, 9, 17,
0, -73, -49, 1, 40, 129, 189,
13, 2, -8, 0, 105, -6, 22, 27, 1, 30, -7, -3, 55, 25, 19,

]

default_ksattackw = [
   0.6662,  0.3207,  1.0303,  1.6283,  1.2155,  0.9181,  1.7826,  1.3251, 0.7895,  1.2036, -9.0123,  0.5335,
   -1.5944e+00, -2.0080e-01,  6.9163e-02, -3.7239e-02, -2.0947e+00, -4.2067e-01,  5.5673e-03,  1.3893e-01,  3.6155e-03,  1.8985e+00, -1.5123e+01,  9.8926e-02
]

mgpoly = [0.2468, 7.1468,-148.4435]
mgscale = 516.9013
egpoly = [-0.5195, 17.5522, 212.7337]
egscale = 901.6464

In [ ]:
default = torch.tensor(default, dtype=torch.float32).view(1, -1)
default_ksattackw = torch.tensor(default_ksattackw, dtype=torch.float32).view(1, -1)
print(default.shape)
print(default_ksattackw.shape)

In [ ]:
class FastTensorDataLoader: # This is the most beautiful thing I have ever seen
    """
    A DataLoader-like object for a set of tensors that can be much faster than
    TensorDataset + DataLoader because dataloader grabs individual indices of
    the dataset and calls cat (slow).
    Source: https://discuss.pytorch.org/t/dataloader-much-slower-than-manual-batching/27014/6
    """
    def __init__(self, *tensors, batch_size=32, shuffle=False):
        """
        Initialize a FastTensorDataLoader.

        :param *tensors: tensors to store. Must have the same length @ dim 0.
        :param batch_size: batch size to load.
        :param shuffle: if True, shuffle the data *in-place* whenever an
            iterator is created out of this object.

        :returns: A FastTensorDataLoader.
        """
        assert all(t.shape[0] == tensors[0].shape[0] for t in tensors)
        self.tensors = tensors

        self.dataset_len = self.tensors[0].shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Calculate # batches
        n_batches, remainder = divmod(self.dataset_len, self.batch_size)
        if remainder > 0:
            n_batches += 1
        self.n_batches = n_batches
    def __iter__(self):
        if self.shuffle:
            r = torch.randperm(self.dataset_len)
            self.tensors = [t[r] for t in self.tensors]
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        batch = tuple(t[self.i:self.i+self.batch_size] for t in self.tensors)
        self.i += self.batch_size
        return batch

    def __len__(self):
        return self.n_batches

In [ ]:
class DangerFormula(nn.Module):
    def __init__(self):
        super(DangerFormula, self).__init__()
        self.scale = nn.Parameter(torch.tensor(512, dtype=torch.float32), requires_grad=True)
        self.K = nn.Parameter(torch.tensor(1 / 64, dtype=torch.float32), requires_grad=False)
        self.linweights = nn.Linear(12, 1, bias=False)
        self.a = nn.Parameter(torch.tensor(0, dtype=torch.float32))
        self.b = nn.Parameter(torch.tensor(1, dtype=torch.float32))
        self.c = nn.Parameter(torch.tensor(-1, dtype=torch.float32))
    
    def forward(self, x):
        x = self.linweights(x)
        return F.sigmoid(
            self.K * (
                self.a * (x**2)
                + self.b * x
                + self.c
            )
        ) * self.scale

# Create the model.
class HCE(nn.Module):
    def __init__(self):
        super(HCE, self).__init__()
        self.mgfc = nn.Linear(NUM_COLORLESS, 1, bias=False)
        self.egfc = nn.Linear(NUM_COLORLESS, 1, bias=False)
        
        self.mgdanger = DangerFormula()
        self.egdanger = DangerFormula()

        self.K = 0.005
    
    def load_default(self):
        # Default is exactly NUM_FEATURES * 2 long, since it contains both the mg and eg values.
        self.mgfc.weight.data = default[:, :NUM_COLORLESS].reshape(1, -1)
        self.egfc.weight.data = default[:, NUM_COLORLESS:].reshape(1, -1)

        self.mgdanger.linweights.weight.data = default_ksattackw[:, :ADDITIONAL_COLORED_FEATS//2].reshape(1,-1)
        self.egdanger.linweights.weight.data = default_ksattackw[:, ADDITIONAL_COLORED_FEATS//2:].reshape(1,-1)

        self.mgdanger.scale.data = torch.tensor(mgscale)
        self.egdanger.scale.data = torch.tensor(egscale)

        self.mgdanger.a.data = torch.tensor(mgpoly[0])
        self.mgdanger.b.data = torch.tensor(mgpoly[1])
        self.mgdanger.c.data = torch.tensor(mgpoly[2])

        self.egdanger.a.data = torch.tensor(egpoly[0])
        self.egdanger.b.data = torch.tensor(egpoly[1])
        self.egdanger.c.data = torch.tensor(egpoly[2])


    def forward(self, phase, colorless, colored):
        colorless = colorless.to(torch.float32)
        colored = colored.to(torch.float32)
        phase = phase.to(torch.float32)
        # King danger index stuff
        white = colored[:, :ADDITIONAL_COLORED_FEATS//2]
        black = colored[:, ADDITIONAL_COLORED_FEATS//2:]

        whitemg = self.mgdanger(white)
        blackmg = self.mgdanger(black)
        whiteeg = self.egdanger(white)
        blackeg = self.egdanger(black)

        dangerTotal = (whitemg - blackmg) * phase + (whiteeg - blackeg) * (1-phase)

        # Calculate the mg and eg values.
        mg = self.mgfc(phase * colorless)
        eg = self.egfc((1-phase)*colorless)
        # Calculate the sigmoid.
        return F.sigmoid(self.K * (mg + eg + dangerTotal))

In [ ]:
def load_model(ptfile):
    model = HCE()
    model.load_state_dict(torch.load(ptfile))
    return model


In [ ]:
# Create the model and load the default values.
m = HCE()
load_epoch = 260
if load_epoch >= 0:
    m = load_model(f"runs/model_K.005_{load_epoch}.pt")
else:
    m.load_default()

In [ ]:
# Create the dataset and the dataloader. Use large batch size to speed up training and avoid overfitting (this is a very simple problem).
BATCH_SIZE = 16384 * 64
train_loader = FastTensorDataLoader(phases_train, colorless_train, colored_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = FastTensorDataLoader(phases_test, colorless_test, colored_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

model = torch.compile(m)

# Create the optimizer.
optimizer = optim.AdamW(model.parameters(), lr=.1, weight_decay=0.001)

# Create the loss function. The original paper uses MSE.
criterion = nn.MSELoss()

# Create the scheduler.
scheduler = optim.lr_scheduler.MultiStepLR(optimizer,milestones=[8, 16, 32, 64, 128, 256, 512, 768, 1024, 1280, 1530, 1786 ], gamma = 0.9)

# Train the model.
EPOCHS = 1024

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Detected device {}".format(device))
model.to(device)
criterion.to(device)

In [ ]:
for epoch in range(load_epoch if load_epoch >= 0 else 0, 2048):
    model.train()
    epoch_start = time.time()
    for phase, colorless, colored, y in train_loader:
        y = y.to(torch.float32)
        phase, colorless, colored, y = phase.to(device, non_blocking = True), colorless.to(device, non_blocking = True), colored.to(device, non_blocking = True), y.to(device, non_blocking = True)
        for param in model.parameters():
            param.grad = None
        y_hat = model(phase, colorless, colored)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
    scheduler.step(epoch)
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for phase, colorless, colored, y in test_loader:
            y = y.to(torch.float32)
            phase, colorless, colored, y = phase.to(device, non_blocking = True), colorless.to(device, non_blocking = True), colored.to(device, non_blocking = True), y.to(device, non_blocking = True)
            y_hat = model(phase, colorless, colored)
            loss = criterion(y_hat, y)
            running_loss += loss.item()

    # Save the model.
    if epoch % 10 == 0:
        torch.save(m.state_dict(), f"runs/model_K.005_{epoch}.pt")

    epoch_end = time.time()
    print(f"(Test) Epoch {epoch}, Loss: {running_loss / len(test_loader)} Time: {epoch_end - epoch_start} s")


In [ ]:
import matplotlib.pyplot as plt
def visualize(m):
    model = m.cpu()
    # Visualize the model. The weights are divided in mg and eg.
    # The first 5 weights are the material values.
    # Then, 64 * 6 weights are the piece square tables.
    # Then 9, 14, 15 and 28 piece mobility features.
    # Visualize these weights using matplotlib and heatmaps.
    weights = model.mgfc.weight.data[0]
    # Material values.
    material = weights[:5]
    # Piece square tables.
    pst = weights[5:5+64*6].reshape(6, 64)
    # Piece mobility features.
    knight_mobility = weights[5+64*6:5+64*6+9]
    bishop_mobility = weights[5+64*6+9:5+64*6+9+14]
    rook_mobility = weights[5+64*6+9+14:5+64*6+9+14+15]
    queen_mobility = weights[5+64*6+9+14+15:5+64*6+9+14+15+28]
    # Create the plots. First, plot the psqt. To do this, we need to:
    # Add the material values to the pst (excluding the king).
    pst[0, :] += material[0]
    pst[1, :] += material[1]
    pst[2, :] += material[2]
    pst[3, :] += material[3]
    pst[4, :] += material[4]
    # Now plot the pst.
    fig, axs = plt.subplots(2, 3)
    # Make the figure bigger using figure() method.
    fig.set_size_inches(18.5, 10.5)
    fig.suptitle("Piece Square Tables")
    for i in range(6):
        axs[i//3, i%3].imshow(pst[i].reshape(8, 8), cmap='hot', interpolation='nearest')
        axs[i//3, i%3].set_title("Piece: " + str(i))
        # Add the value text to the plot.
        for y in range(8):
            for x in range(8):
                axs[i//3, i%3].text(x, y, str(round(pst[i, y*8 + x].item())), color='black', ha='center', va='center')
    plt.show()
    # Now show the mobility features. For each piece, plot the mobility features using a bar plot.
    fig, axs = plt.subplots(2, 2)
    fig.set_size_inches(12.5, 6.5)
    fig.suptitle("Piece Mobility Features")
    axs[0, 0].bar(range(9), knight_mobility)
    axs[0, 0].set_title("Knight Mobility")
    axs[0, 1].bar(range(14), bishop_mobility)
    axs[0, 1].set_title("Bishop Mobility")
    axs[1, 0].bar(range(15), rook_mobility)
    axs[1, 0].set_title("Rook Mobility")
    axs[1, 1].bar(range(28), queen_mobility)
    axs[1, 1].set_title("Queen Mobility")   
    # Add the value text to the plot.
    for i in range(9):
        axs[0, 0].text(i, knight_mobility[i], str(round(knight_mobility[i].item())), color='black', ha='center', va='bottom')
    for i in range(14):
        axs[0, 1].text(i, bishop_mobility[i], str(round(bishop_mobility[i].item())), color='black', ha='center', va='bottom')
    for i in range(15):
        axs[1, 0].text(i, rook_mobility[i], str(round(rook_mobility[i].item())), color='black', ha='center', va='bottom')
    for i in range(28):
        axs[1, 1].text(i, queen_mobility[i], str(round(queen_mobility[i].item())), color='black', ha='center', va='bottom')

    # Now for eg.
    weights = model.egfc.weight.data[0]
    # Material values.
    material = weights[:5]
    # Piece square tables.
    pst = weights[5:5+64*6].reshape(6, 64)
    # Piece mobility features.
    knight_mobility = weights[5+64*6:5+64*6+9]
    bishop_mobility = weights[5+64*6+9:5+64*6+9+14]
    rook_mobility = weights[5+64*6+9+14:5+64*6+9+14+15]
    queen_mobility = weights[5+64*6+9+14+15:5+64*6+9+14+15+28]
    # Create the plots. First, plot the psqt. To do this, we need to:
    # Add the material values to the pst (excluding the king).
    pst[0, :] += material[0]
    pst[1, :] += material[1]
    pst[2, :] += material[2]
    pst[3, :] += material[3]
    pst[4, :] += material[4]
    # Now plot the pst.
    fig, axs = plt.subplots(2, 3)
    # Make the figure bigger using figure() method.
    fig.set_size_inches(18.5, 10.5)
    fig.suptitle("Piece Square Tables")
    for i in range(6):
        axs[i//3, i%3].imshow(pst[i].reshape(8, 8), cmap='hot', interpolation='nearest')
        axs[i//3, i%3].set_title("Piece: " + str(i))
        # Add the value text to the plot.
        for y in range(8):
            for x in range(8):
                axs[i//3, i%3].text(x, y, str(round(pst[i, y*8 + x].item())), color='black', ha='center', va='center')
    plt.show()
    # Now show the mobility features. For each piece, plot the mobility features using a bar plot.
    fig, axs = plt.subplots(2, 2)
    fig.set_size_inches(12.5, 6.5)
    fig.suptitle("Piece Mobility Features")
    axs[0, 0].bar(range(9), knight_mobility)
    axs[0, 0].set_title("Knight Mobility")
    axs[0, 1].bar(range(14), bishop_mobility)
    axs[0, 1].set_title("Bishop Mobility")
    axs[1, 0].bar(range(15), rook_mobility)
    axs[1, 0].set_title("Rook Mobility")
    axs[1, 1].bar(range(28), queen_mobility)
    axs[1, 1].set_title("Queen Mobility")
    # Add the value text to the plot.
    for i in range(9):
        axs[0, 0].text(i, knight_mobility[i], str(round(knight_mobility[i].item())), color='black', ha='center', va='bottom')
    for i in range(14):
        axs[0, 1].text(i, bishop_mobility[i], str(round(bishop_mobility[i].item())), color='black', ha='center', va='bottom')
    for i in range(15):
        axs[1, 0].text(i, rook_mobility[i], str(round(rook_mobility[i].item())), color='black', ha='center', va='bottom')
    for i in range(28):
        axs[1, 1].text(i, queen_mobility[i], str(round(queen_mobility[i].item())), color='black', ha='center', va='bottom')
    plt.show()
    
visualize(model)

In [ ]:
# Print the final weights. Print the weights rounded to 2 decimal places.
p = 5+64*6+9+14+15+28
newliners = [
    5, 5+64*1,5+64*2,5+64*3,5+64*4,5+64*5,5+64*6, 5+64*6+9, 5+64*6+9+14, 5+64*6+9+14+15, 5+64*6+9+14+15+28,
    p+1, p+2, p+3, p+4, p+5, p+6, p+7, p+8,
    p+15, p+16, p+17, p+18, p+19, p+20, p+21, p+22, p+23, p+24,p+25,p+26,p+27,p+28,p+29,
]



def psqtprint(numbers, s):# Determine the maximum width of the numbers
    max_width = 4
    
    # Print the numbers with proper formatting
    for i in range(0, len(numbers), 8):
        row = numbers[i:i+8]
        formatted_row = ', '.join(f'{round(num - s):>{max_width}}' for num in row)
        print(f'    {formatted_row} ,')


weights = model.mgfc.weight.data[0]
stuff = []
s = []
for i in range(len(weights)):
    # print(f"{round(weights[i].item())}", end=", ")
    s.append(weights[i].item())
    if i+1 in newliners:
        # print()
        stuff.append(s)
        s = []
stuff.append(s)

heads = ["const Score","constexpr Score"]

names = ["mgValues", "mgPawnTable", "mgKnightTable", "mgBishopTable", "mgRookTable", "mgQueenTable", "mgKingTable", "knightMobMg", "bishopMobMg", "rookMobMg", "queenMobMg",
         "DOUBLEISOLATEDPENMG", "ISOLATEDPENMG", "BACKWARDPENMG", "DOUBLEDPENMG", "SUPPORTEDPHALANXMG", "ADVANCABLEPHALANXMG", "R_SUPPORTEDPHALANXMG", "R_ADVANCABLEPHALANXMG",
         "passedRankBonusMg", "PASSEDPATHBONUSMG", "SUPPORTEDPASSERMG", "INNERSHELTERMG", "OUTERSHELTERMG", "BISHOPPAIRMG", "ROOKONOPENFILEMG", "ROOKONSEMIOPENFILEMG","KNIGHTONEXTOUTPOSTMG","BISHOPONEXTOUTPOSTMG","KNIGHTONINTOUTPOSTMG","BISHOPONINTOUTPOSTMG", "BISHOPPAWNSMG", "THREATSAFEPAWNMG","THREATPAWNPUSHMG","TEMPOMG"]

for i in range(len(stuff)):
    item = stuff[i]    
    if len(item)==64:
        print(heads[0]+" "+names[i]+" ["+str(len(item))+"] = {\n")
        psqtprint(item, stuff[0][i-1] if i < 6 else 0)
        print("\n};")
    elif len(item)>1:
        print(heads[0]+" "+names[i]+" ["+str(len(item))+"] = {",end = "")
        for x in item:
            print(round(x),end=", ")
        print("};")
    else:
        print(heads[1]+" "+names[i]+" = "+str(round(item[0]))+";")


data1 = model.mgdanger.smolnetA.weight.data
bias1 = model.mgdanger.smolnetA.bias.data
print("Mg danger data:",data1, bias1)
#weights = model.mg_attack_weights.weight.data
#bias = model.mg_attack_weights.bias.data
#print("Mg attack weights ( x",1/model.mgdanger.K.data,")", weights, "bias", bias)
        

weights = model.egfc.weight.data[0]
stuff = []
s = []
for i in range(len(weights)):
    # print(f"{round(weights[i].item())}", end=", ")
    s.append(weights[i].item())
    if i+1 in newliners:
        # print()
        stuff.append(s)
        s = []
stuff.append(s)

heads = ["const Score","constexpr Score"]

nameseg = ["egValues", "egPawnTable", "egKnightTable", "egBishopTable", "egRookTable", "egQueenTable", "egKingTable", "knightMobEg", "bishopMobEg", "rookMobEg", "queenMobEg",
         "DOUBLEISOLATEDPENEG", "ISOLATEDPENEG", "BACKWARDPENEG", "DOUBLEDPENEG", "SUPPORTEDPHALANXEG", "ADVANCABLEPHALANXEG", "R_SUPPORTEDPHALANXEG", "R_ADVANCABLEPHALANXEG",
         "passedRankBonusEg", "PASSEDPATHBONUSEG", "SUPPORTEDPASSEREG", "INNERSHELTEREG", "OUTERSHELTEREG", "BISHOPPAIREG", "ROOKONOPENFILEEG", "ROOKONSEMIOPENFILEEG","KNIGHTONEXTOUTPOSTEG","BISHOPONEXTOUTPOSTEG","KNIGHTONINTOUTPOSTEG","BISHOPONINTOUTPOSTEG", "BISHOPPAWNSEG", "THREATSAFEPAWNEG","THREATPAWNPUSHEG", "TEMPOEG"]

for i in range(len(stuff)):
    item = stuff[i]    
    if len(item)==64:
        print(heads[0]+" "+nameseg[i]+" ["+str(len(item))+"] = {\n")
        psqtprint(item, stuff[0][i-1] if i < 6 else 0)
        print("\n};")
    elif len(item)>1:
        print(heads[0]+" "+nameseg[i]+" ["+str(len(item))+"] = {",end = "")
        for x in item:
            print(round(x),end=", ")
        print("};")
    else:
        print(heads[1]+" "+nameseg[i]+" = "+str(round(item[0]))+";")
